# OSM Scene Generation: Obelisco de Buenos Aires
## Urban Geometry Visualization for Passive Radar Simulation

This notebook demonstrates the extraction and 3D visualization of urban geometry using OpenStreetMap (OSM). It focuses on the area surrounding the **Obelisco de Buenos Aires**.

In [1]:
#@title ── 0. Environment Setup { display-mode: "form" }
import os, sys, subprocess, importlib.util

REPO_NAME = "multipath-passive-radar-sim"
REPO_PATH = f"/content/{REPO_NAME}"

# Clone repo if needed
if not os.path.exists(REPO_PATH):
    print("Cloning repository...")
    subprocess.run(
        ["git","clone",f"https://github.com/dealmeidapaulo/{REPO_NAME}.git",REPO_PATH],
        capture_output=True,
    )

%cd {REPO_PATH}

SRC_PATH = os.path.join(REPO_PATH, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Install editable package
subprocess.run(["pip","install","-e",".","-q"], capture_output=True)
subprocess.run(["pip","install","osmnx","pyproj","shapely"], capture_output=True)

# Core imports
import numpy as np
import plotly.graph_objects as go
import src.core.scene.osm as osm
# GPU check
r = subprocess.run(
    ["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
    capture_output=True, text=True,
)
print(f"GPU: {r.stdout.strip() if r.returncode==0 else 'None -- change runtime to T4'}")

Cloning repository...
/content/multipath-passive-radar-sim
GPU: Tesla T4, 15360 MiB


In [3]:
#@title ── 1. Scene Generation & Parameters ─────────────────────────────

LAT = -34.60373 #@param {type:"number"}
LON = -58.38157 #@param {type:"number"}
DIST_M = 500 #@param {type:"number"}

print(f"Fetching extended OSM data for LAT: {LAT}, LON: {LON}...")

scene = osm.load_osm_obstacles(
    lat=LAT,
    lon=LON,
    dist_m=DIST_M
)

print(f"Scene generated with {len(scene.obstacles)} obstacles.")

Fetching extended OSM data for LAT: -34.60373, LON: -58.38157...
Scene generated with 852 obstacles.


In [4]:
#@title ── 2. 3D Visualization { display-mode: "form" }
import plotly.graph_objects as go

def plot_urban_geometry(scene):
    fig = go.Figure()

    for obs in scene.obstacles:
        # We use MeshObstacle properties (vertices and faces)
        # The current osm.py generates MeshObstacles
        if hasattr(obs, 'vertices') and hasattr(obs, 'faces'):
            v = obs.vertices
            f = obs.faces

            fig.add_trace(go.Mesh3d(
                x=v[:, 0], y=v[:, 1], z=v[:, 2],
                i=f[:, 0], j=f[:, 1], k=f[:, 2],
                opacity=0.7,
                color='#555555',
                flatshading=True,
                name=obs.material,
                lighting=dict(ambient=0.4, diffuse=0.8)
            ))

    fig.update_layout(
        template="plotly_dark",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            camera=dict(
                eye=dict(x=0.0, y=0.0, z=2.0), # Aerial / Nadir view
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=1, z=0)
            )
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        title=dict(
            text="Scene",
            x=0.5, y=0.95, font=dict(size=16, color="#ffffff")
        )
    )
    fig.show()

plot_urban_geometry(scene)